# Module 7 Guided Lab: Aggregation, Reshaping, and Light Validation

**BAN 6003: Data Management and Analytics Integration**

In this lab, we return to the `nycflights13` data. This module is practical: we will use Pandas to summarize data, reshape data, and run light validation checks after those transformations.

The key idea for this week is:

> Aggregation and reshaping can change what each row means.

The original `flights` table has one row per flight. After transformation, a row might mean one carrier, one month, one route, or one carrier-month combination.

## Lab Learning Goals

By the end of this lab, you should be able to:

1. Use `groupby()` and `agg()` to summarize flight data.
2. Use `reset_index()` to turn grouped results back into a regular DataFrame.
3. Explain how aggregation changes the unit of analysis.
4. Use `melt()` to reshape wide data into long data.
5. Use `pivot()` or `pivot_table()` to reshape long data into wide data.
6. Run light validation checks after aggregation or reshaping.
7. Write short notes explaining whether the transformed dataset still matches the intended unit of analysis.

We will not use custom functions or complex loops.

## Business Scenario

Imagine you are an analyst helping an airport operations team understand flight delays.

The raw `flights` table has one row per flight. That is useful if we want to inspect individual flights.

But managers usually ask higher-level questions:

- Which carrier has the highest average departure delay?
- Which month has the worst average arrival delay?
- Which origin-destination route has the most flights?
- How do monthly delay patterns compare across carriers?

To answer these questions, we need aggregation and reshaping.

## 0. Setup: Load Packages and Data

The repository requirements include `nycflights13`. In Codespaces they are installed during setup. When working locally, install `requirements.txt` before opening the notebook.

### Reproducibility Note

Package installation is kept outside the analytical workflow. This makes the notebook easier to rerun and avoids hidden environment changes in the middle of an analysis.

In [1]:
import nycflights13
import pandas as pd
from pathlib import Path

/opt/conda/lib/python3.12/site-packages/nycflights13/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename as _rf


In [2]:
flights = nycflights13.flights.copy()
flights.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01T10:00:00Z
1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01T10:00:00Z
2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01T10:00:00Z
3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01T10:00:00Z
4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01T11:00:00Z


### First reminder: What does one row mean?

Before transforming anything, we need to know the current unit of analysis. In the original `flights` table, each row represents one flight.

In [3]:
flights.shape

(336776, 19)

In [4]:
flights.columns

Index(['year', 'month', 'day', 'dep_time', 'sched_dep_time', 'dep_delay',
       'arr_time', 'sched_arr_time', 'arr_delay', 'carrier', 'flight',
       'tailnum', 'origin', 'dest', 'air_time', 'distance', 'hour', 'minute',
       'time_hour'],
      dtype='str')

In [5]:
flights[["year", "month", "day", "carrier", "flight", "origin", "dest", "dep_delay", "arr_delay", "distance", "air_time"]].head()

,year,month,day,carrier,flight,origin,dest,dep_delay,arr_delay,distance,air_time
0,2013,1,1,UA,1545,EWR,IAH,2.0,11.0,1400,227.0
1,2013,1,1,UA,1714,LGA,IAH,4.0,20.0,1416,227.0
2,2013,1,1,AA,1141,JFK,MIA,2.0,33.0,1089,160.0
3,2013,1,1,B6,725,JFK,BQN,-1.0,-18.0,1576,183.0
4,2013,1,1,DL,461,LGA,ATL,-6.0,-25.0,762,116.0


## 1. Aggregation with `groupby()` and `agg()`

### Business question

Which airline carrier had the highest average departure delay?

To answer this, we need to move from one row per flight to one row per carrier. That is aggregation.

The basic pattern is:

```python
data.groupby("grouping_column").agg({"value_column": "summary_function"})
```

Important arguments:

- The column inside `groupby()` defines the level of the summary.
- The dictionary inside `agg()` says which variable to summarize and how.
- Common summary functions include `"mean"`, `"median"`, `"count"`, `"max"`, `"min"`, and `"std"`.

In [6]:
carrier_delay = flights.groupby("carrier").agg({
    "dep_delay": "mean"
})

carrier_delay.head()

,dep_delay
carrier,
9E,16.725769
AA,8.586016
AS,5.804775
B6,13.022522
DL,9.264505


The result is indexed by `carrier`. This is useful, but it does not look like a regular DataFrame yet. In many workflows, we use `reset_index()` after a grouped summary.

In [7]:
carrier_delay = flights.groupby("carrier").agg({
    "dep_delay": "mean"
}).reset_index()

carrier_delay.head()

,carrier,dep_delay
0,9E,16.725769
1,AA,8.586016
2,AS,5.804775
3,B6,13.022522
4,DL,9.264505


Now we can sort the summary to see which carriers have the highest average departure delay.

In [8]:
carrier_delay.sort_values("dep_delay", ascending=False).head()

,carrier,dep_delay
6,F9,20.215543
5,EV,19.955390
15,YV,18.996330
7,FL,18.726075
14,WN,17.711744


### What changed?

The original data had one row per flight. The aggregated data has one row per carrier.

That means `dep_delay` no longer means "delay for one flight." It now means "average departure delay for that carrier."

This is why aggregation changes the meaning of a row.

### Your Turn 1

Which carrier had the lowest average arrival delay?

Use `groupby()`, `agg()`, `reset_index()`, and `sort_values()`.

In [9]:
carrier_delay = (
    flights
    .groupby("carrier")
    .agg(avg_arr_delay=("arr_delay", "mean"))
    .reset_index()
    .sort_values("avg_arr_delay", ascending=True)
)

carrier_delay

,carrier,avg_arr_delay
2,AS,-9.930889
8,HA,-6.915205
1,AA,0.364291
4,DL,1.644341
13,VX,1.764464
12,US,2.129595
11,UA,3.558011
0,9E,7.379669
3,B6,9.457973
14,WN,9.649120


Write one sentence explaining what one row means in your result.

**Your answer:**  
Each row represents a carrier and its average delay across all flights operated by that carrier

## 2. Multiple Aggregations

### Business question

For each carrier, what is the average departure delay, the maximum departure delay, and the number of flights?

This is useful because an average alone can hide important operational risk. A carrier may have a moderate average delay but still have some extreme delay cases.

In [10]:
carrier_summary = flights.groupby("carrier").agg(
    avg_dep_delay=("dep_delay", "mean"),
    max_dep_delay=("dep_delay", "max"),
    flight_count=("flight", "count")
).reset_index()

carrier_summary.head()

,carrier,avg_dep_delay,max_dep_delay,flight_count
0,9E,16.725769,747.0,18460
1,AA,8.586016,1014.0,32729
2,AS,5.804775,225.0,714
3,B6,13.022522,502.0,54635
4,DL,9.264505,960.0,48110


This syntax is called named aggregation. It lets us choose clear output column names.

The pattern is:

```python
new_column_name=("original_column", "summary_function")
```

This is easier to read than a table with multi-level column names.

In [11]:
carrier_summary.sort_values("avg_dep_delay", ascending=False).head(10)

,carrier,avg_dep_delay,max_dep_delay,flight_count
6,F9,20.215543,853.0,685
5,EV,19.955390,548.0,54173
15,YV,18.996330,387.0,601
7,FL,18.726075,602.0,3260
14,WN,17.711744,471.0,12275
0,9E,16.725769,747.0,18460
3,B6,13.022522,502.0,54635
13,VX,12.869421,653.0,5162
10,OO,12.586207,154.0,32
11,UA,12.106073,483.0,58665


### Your Turn 2

Create a monthly delay summary with one row per month.

Your result should include:

- average arrival delay
- median arrival delay
- number of flights

Then sort by average arrival delay from highest to lowest.

In [12]:
monthly_summary = (
    flights.groupby("month")
    .agg(
        avg_arr_delay=("arr_delay", "mean"),
        median_arr_delay=("arr_delay", "median"),
        flight_count=("arr_delay", "count")
    )
    .sort_values("avg_arr_delay", ascending=False)
)

monthly_summary


,avg_arr_delay,median_arr_delay,flight_count
month,,,
7,16.711307,-2.0,28293
6,16.481330,-2.0,27075
12,14.870355,2.0,27020
4,11.176063,-2.0,27564
1,6.129972,-3.0,26398
8,6.040652,-5.0,28756
3,5.807577,-6.0,27902
2,5.613019,-3.0,23611
5,3.521509,-8.0,28128


Write one sentence explaining what one row means in your monthly summary.

**Your answer:**  
Each row represents a month and shows the average arrival delay, median arrival delay, and total number of flights for that month.

## 3. Aggregating at Two Levels: Carrier and Month

### Business question

Do some carriers experience worse delays in certain months?

Now we want one row per carrier-month combination. This is a different unit of analysis from one row per carrier or one row per month.

In [13]:
carrier_month_delay = flights.groupby(["carrier", "month"]).agg(
    avg_arr_delay=("arr_delay", "mean"),
    flight_count=("flight", "count")
).reset_index()

carrier_month_delay.head()

,carrier,month,avg_arr_delay,flight_count
0,9E,1,10.207432,1573
1,9E,2,8.283371,1459
2,9E,3,2.031788,1627
3,9E,4,5.471051,1511
4,9E,5,10.415763,1462


Here, the grouping columns are `carrier` and `month`.

One row now means:

> one carrier in one month

This type of table is often the starting point for dashboards and business reports.

In [14]:
carrier_month_delay.sort_values("avg_arr_delay", ascending=False).head(10)

,carrier,month,avg_arr_delay,flight_count
120,OO,1,107.000000,1
121,OO,6,68.500000,2
122,OO,8,64.500000,4
90,FL,7,44.967742,263
178,YV,6,44.589744,49
89,FL,6,41.966805,252
78,F9,7,36.413793,58
75,F9,4,34.315789,57
73,F9,2,31.145833,49
176,YV,4,30.972222,38


### Light validation after aggregation

After aggregation, we should check the basic structure of the result.

For this table, each carrier-month combination should appear only once.

In [15]:
carrier_month_delay.shape

(185, 4)

In [17]:
carrier_month_delay[["carrier", "month"]].duplicated().sum()

np.int64(0)

In [18]:
carrier_month_delay.isna().sum()

carrier          0
month            0
avg_arr_delay    0
flight_count     0
dtype: int64

These are light validation checks. We are not doing a full final data audit yet. We are simply checking whether the aggregation result has the structure we intended.

### Your Turn 3

Create a route-level summary with one row per `origin` and `dest`.

Your result should include:

- number of flights
- average arrival delay
- average distance

Then sort to find the busiest routes.

In [19]:
route_summary = (
    flights.groupby(["origin", "dest"])
    .agg(
        flight_count=("arr_delay", "count"),
        avg_arr_delay=("arr_delay", "mean"),
        avg_distance=("distance", "mean")
    )
    .sort_values("flight_count", ascending=False)
)

route_summary

flight_count  avg_arr_delay  avg_distance
origin dest                                           
JFK    LAX          11159      -0.480599        2475.0
LGA    ATL          10041      11.322478         762.0
       ORD           8507       1.833431         733.0
JFK    SFO           8109       2.879640        2586.0
LGA    CLT           5961       7.379802         544.0
...                   ...            ...           ...
JFK    BHM              1     -19.000000         865.0
       STL              1      -8.000000         892.0
       MEM              1      -5.000000         964.0
LGA    LEX              1     -22.000000         604.0
EWR    LGA              0            NaN          17.0

[224 rows x 3 columns]

Run a light validation check: do any origin-destination pairs appear more than once in your route summary?

In [20]:
route_summary.reset_index()[["origin", "dest"]].duplicated().sum()

np.int64(0)

## 4. Wide vs Long Data

Aggregation summarizes data. Reshaping changes how the same values are arranged.

A wide table uses multiple columns to store different categories or time periods. A long table stores those categories or time periods in a variable column.

Both formats can be useful.

- Wide data is often easier for people to read in a report.
- Long data is often easier for plotting, grouping, and modeling workflows.

## 5. Reshaping Wide to Long with `melt()`

### Business question

Suppose a manager gives us a small report where each month is stored as a separate column. How can we reshape it so each row represents one carrier-month?

We will create a small wide example using carrier-level monthly average arrival delays.

In [21]:
monthly_carrier = flights.groupby(["carrier", "month"]).agg(
    avg_arr_delay=("arr_delay", "mean")
).reset_index()

monthly_carrier.head()

,carrier,month,avg_arr_delay
0,9E,1,10.207432
1,9E,2,8.283371
2,9E,3,2.031788
3,9E,4,5.471051
4,9E,5,10.415763


In [22]:
small_summary = monthly_carrier.query("carrier in ['AA', 'DL', 'UA'] and month in [1, 2, 3]")
small_summary

,carrier,month,avg_arr_delay
12,AA,1,0.982379
13,AA,2,1.093789
14,AA,3,-2.976651
48,DL,1,-4.404651
49,DL,2,-4.726007
50,DL,3,1.414888
125,UA,1,3.175599
126,UA,2,0.194611
127,UA,3,1.553066


In [23]:
wide_delay = small_summary.pivot(
    index="carrier",
    columns="month",
    values="avg_arr_delay"
).reset_index()

wide_delay

month,carrier,1,2,3
0,AA,0.982379,1.093789,-2.976651
1,DL,-4.404651,-4.726007,1.414888
2,UA,3.175599,0.194611,1.553066


This wide table is easy to read: each row is a carrier, and each month is a separate column.

But if we want to plot month as a variable or group by month again, long format is usually better.

Now we use `melt()`.

In [24]:
long_delay = wide_delay.melt(
    id_vars="carrier",
    var_name="month",
    value_name="avg_arr_delay"
)

long_delay

,carrier,month,avg_arr_delay
0,AA,1,0.982379
1,DL,1,-4.404651
2,UA,1,3.175599
3,AA,2,1.093789
4,DL,2,-4.726007
5,UA,2,0.194611
6,AA,3,-2.976651
7,DL,3,1.414888
8,UA,3,1.553066


Important arguments in `melt()`:

- `id_vars`: the column or columns that should stay fixed.
- `var_name`: the name of the new column that stores the old column names.
- `value_name`: the name of the new column that stores the old cell values.

After melting, one row means one carrier-month.

### Your Turn 4

Use `melt()` on `wide_delay`.

Create a new long DataFrame called `long_delay_practice`.

Use:

- `carrier` as the ID variable
- `delay_month` as the variable name
- `average_delay` as the value name

In [25]:
long_delay_practice = wide_delay.melt(
    id_vars="carrier",
    var_name="delay_month",
    value_name="average_delay"
)

long_delay_practice


,carrier,delay_month,average_delay
0,AA,1,0.982379
1,DL,1,-4.404651
2,UA,1,3.175599
3,AA,2,1.093789
4,DL,2,-4.726007
5,UA,2,0.194611
6,AA,3,-2.976651
7,DL,3,1.414888
8,UA,3,1.553066


Run a quick check: how many rows should the long dataset have, based on the number of carriers and month columns?

In [26]:
len(wide_delay) * (len(wide_delay.columns) - 1)

9

## 6. Reshaping Long to Wide with `pivot()` and `pivot_table()`

### Business question

A manager wants a compact comparison table with one row per carrier and one column per month. How do we move from long back to wide?

We can use `pivot()` when each index-column combination has only one value.

In [27]:
wide_again = long_delay.pivot(
    index="carrier",
    columns="month",
    values="avg_arr_delay"
).reset_index()

wide_again

month,carrier,1,2,3
0,AA,0.982379,1.093789,-2.976651
1,DL,-4.404651,-4.726007,1.414888
2,UA,3.175599,0.194611,1.553066


Important arguments in `pivot()`:

- `index`: what should define each row.
- `columns`: which values should become new columns.
- `values`: which numeric values should fill the table.

If there are duplicate combinations, `pivot()` may fail. In that case, `pivot_table()` is often safer because it can aggregate duplicates.

In [28]:
wide_with_pivot_table = small_summary.pivot_table(
    index="carrier",
    columns="month",
    values="avg_arr_delay",
    aggfunc="mean"
).reset_index()

wide_with_pivot_table

month,carrier,1,2,3
0,AA,0.982379,1.093789,-2.976651
1,DL,-4.404651,-4.726007,1.414888
2,UA,3.175599,0.194611,1.553066


### Your Turn 5

Use the `carrier_month_delay` table from earlier.

Create a wide table with:

- one row per carrier
- one column per month
- values equal to average arrival delay

Use `pivot_table()`.

In [29]:
wide_with_pivot_table = carrier_month_delay.pivot_table(
    index="carrier",
    columns="month",
    values="avg_arr_delay"
)

wide_with_pivot_table


month,1,2,3,4,5,6,7,8,9,10,11,12
carrier,,,,,,,,,,,,
9E,10.207432,8.283371,2.031788,5.471051,10.415763,22.511905,23.763256,5.313911,-7.143052,-1.345566,-0.029243,14.046967
AA,0.982379,1.093789,-2.976651,8.257833,-4.682066,6.481178,4.317443,-2.304009,-8.573988,-3.794368,-2.017647,7.719677
AS,8.967742,-12.740741,-4.838710,3.500000,-10.796610,-3.750000,-15.725806,-18.096774,-34.900000,-25.919355,-10.807692,7.462963
B6,4.717199,12.479341,10.860936,13.266652,2.716773,18.541319,23.349245,11.316130,-2.514520,-2.968327,-0.876667,19.501718
DL,-4.404651,-4.726007,1.414888,3.444363,-0.390064,13.261829,14.920631,0.832791,-4.414533,-6.295110,-2.760813,6.169904
EV,25.160192,20.269003,20.717456,21.168711,14.174113,21.267975,21.668371,11.010145,-1.870712,9.040108,5.087423,26.773066
F9,21.830508,31.145833,12.666667,34.315789,27.086207,26.636364,36.413793,20.618182,2.285714,9.859649,18.216667,22.983607
FL,3.317901,4.248227,16.132013,19.310231,18.332298,41.966805,44.967742,24.835294,15.071146,15.991453,18.539604,29.105769
HA,27.483871,-28.821429,-17.322581,-17.066667,-21.064516,1.833333,3.000000,4.387097,-19.400000,-14.428571,2.320000,-9.357143


Run a light validation check:

- How many rows does your wide table have?
- Does that match the number of unique carriers?

In [30]:
print("Rows in wide table:", len(wide_with_pivot_table))
print("Unique carriers:", carrier_month_delay["carrier"].nunique())


Rows in wide table: 16
Unique carriers: 16


## 7. Light Validation Checklist

After aggregation or reshaping, always ask:

1. What does one row mean now?
2. Did the number of rows change in a way that makes sense?
3. Are the intended keys unique?
4. Did missing values appear or disappear?
5. Do the column names still clearly describe the values?

These checks are small, but they catch many common mistakes.

In [31]:
long_delay[["carrier", "month"]].duplicated().sum()

np.int64(0)

In [32]:
long_delay.isna().sum()

carrier          0
month            0
avg_arr_delay    0
dtype: int64

In [33]:
long_delay[["carrier", "month"]].drop_duplicates().shape[0]

9

## 8. Final Practice: Carrier-Month Dashboard Table

Create a table named `carrier_month_summary` with one row per carrier-month and:

- `avg_departure_delay`
- `avg_arrival_delay`
- `flight_count`
- `avg_distance`

Then create `carrier_month_summary_sorted` with the highest average arrival delays first. Confirm that carrier-month keys are unique and inspect missing values.

In [36]:
carrier_month_summary = (
    flights.groupby(["carrier", "month"])
    .agg(
        avg_departure_delay=("dep_delay", "mean"),
        avg_arrival_delay=("arr_delay", "mean"),
        flight_count=("arr_delay", "count"),
        avg_distance=("distance", "mean")
    )
    .reset_index()
)


In [35]:
carrier_month_summary_sorted = (
    carrier_month_summary
    .sort_values("avg_arrival_delay", ascending=False)
)

carrier_month_summary_sorted.head()


,carrier,month,avg_departure_delay,avg_arrival_delay,flight_count,avg_distance
120,OO,1,67.000000,107.000000,1,733.000000
121,OO,6,61.000000,68.500000,2,488.000000
122,OO,8,64.000000,64.500000,4,419.000000
90,FL,7,41.162698,44.967742,248,644.034221
178,YV,6,42.794872,44.589744,39,389.714286


In [37]:
carrier_month_summary[["carrier", "month"]].duplicated().sum()


np.int64(0)

In [38]:
carrier_month_summary.isna().sum()


carrier                0
month                  0
avg_departure_delay    0
avg_arrival_delay      0
flight_count           0
avg_distance           0
dtype: int64

In [39]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

if carrier_month_summary is None:
    print("Complete Final Practice Step 1 before exporting.")
else:
    module7_output_path = output_dir / "carrier_month_summary_module7.csv"
    carrier_month_summary.to_csv(module7_output_path, index=False)
    print(f"Saved {len(carrier_month_summary)} rows to: {module7_output_path}")


Saved 185 rows to: ../data/processed/carrier_month_summary_module7.csv


### Final Reflection

Write 3-5 sentences answering the following:

1. What does one row mean in `carrier_month_summary`?
2. Which transformation changed the row meaning the most?
3. What validation evidence supports the table's intended structure?
4. How could this table support an operations decision?

**Your reflection:**  
Each row in carrier_month_summary is a unique carrier-month combination and has average departure delay, average arrival delay, flight count, and average distance. The transformation that changed the row meaning the most was the groupby() aggregation as it converted individual flight records into monthly summaries for each carrier. Validation checks, such as confirming there were no duplicate carrier-month keys and checking for missing values, gave evidence that the table had the intended structure. This table can support operational decisions by helping identify carriers and months with consistently higher delays so resources can be allocated more effectively.

## 9. Offline Assignment - Route Performance Summary

An airport operations manager wants to review established routes rather than individual flights.

1. Create `route_performance` with one row per `origin`-`dest` pair.
2. Include flight count, average arrival delay, and average distance using named aggregation.
3. Keep routes with at least 500 flights and sort from highest to lowest average arrival delay.
4. Verify that route keys are unique and inspect missing values.
5. Save `data/processed/route_performance_module7.csv`.

In [40]:
# Create route_performance
route_performance = (
    flights.groupby(["origin", "dest"], as_index=False)
    .agg(
        flight_count=("arr_delay", "count"),
        avg_arrival_delay=("arr_delay", "mean"),
        avg_distance=("distance", "mean")
    )
)

# Keep routes with at least 500 flights
route_performance = route_performance[
    route_performance["flight_count"] >= 500
]

# Sort from highest to lowest average arrival delay
route_performance = (
    route_performance
    .sort_values("avg_arrival_delay", ascending=False)
    .reset_index(drop=True)
)

# Validation checks
print("Duplicate route keys:",
      route_performance[["origin", "dest"]].duplicated().sum())

print("\nMissing values:")
print(route_performance.isna().sum())

# Save CSV
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

route_performance.to_csv(
    output_dir / "route_performance_module7.csv",
    index=False
)

route_performance.head()



Duplicate route keys: 0

Missing values:
origin               0
dest                 0
flight_count         0
avg_arrival_delay    0
avg_distance         0
dtype: int64


,origin,dest,flight_count,avg_arrival_delay,avg_distance
0,EWR,RIC,1615,25.820433,277.0
1,EWR,MKE,1058,24.003781,725.0
2,EWR,PWM,730,23.005479,284.0
3,JFK,CMH,705,22.920567,483.0
4,EWR,DCA,1567,21.313976,199.0


### Route Summary Interpretation

**Your response here:** Each row is a unique origin-destination route and summarizes the average arrival delay, average distance, and number of flights for that route. The 500-flight filter narrows the analysis to established routes with substantial traffic, which helps reduce the influence of routes with very few observations. Validation checks confirmed that route keys were unique and helped identify any missing values before saving the final table. This summary can support operational decisions by highlighting high-volume routes with higher average delays that may warrant additional monitoring or resource planning.

## 10. Save Your Work

Before submitting:

1. Complete all Your Turn and final-practice cells.
2. Complete the offline route summary and both reflections.
3. Restart the kernel and run all cells from top to bottom.
4. Confirm both Module 7 CSV files exist in `data/processed`.
5. Save the notebook, then commit and push your work.

Suggested commit message:

```bash
git add .
git commit -m "Complete Module 7 aggregation and reshaping lab"
git push
```